# Детекция КП

Два режима:

1. **circle-first** (`detect_controls`) — soft magenta → кольца → EasyOCR рядом.
2. **VLM** (`detect_controls_vlm`) — Yandex AI Studio (Gemma) / OpenAI → JSON → snap к кольцам.

Для VLM из РФ: `YC_API_KEY` + `YC_FOLDER_ID` в `.env` (см. `.env.example`). OpenAI часто даёт 403.


In [ ]:
from pathlib import Path
import sys
import os

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    if (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError("Не найден src/")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src.controls import detect_controls, detect_controls_vlm, draw_controls

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
# Тайл / фото карты:
IMAGE = ROOT / "dataset" / "semenkino_2005_omaps" / "tiles" / "tile_0013" / "image.jpg"
# IMAGE = Path("/path/to/phone_photo.jpg")

PRIOR = IMAGE.parent / "channels.npy"
prior = np.load(PRIOR)[15] if PRIOR.exists() else None

assert IMAGE.exists(), IMAGE
rgb = np.array(Image.open(IMAGE).convert("RGB"))

# circle_first (по умолчанию): кольца → OCR рядом
result = detect_controls(
    rgb,
    prior_mask=prior,
    sensitivity="recall",
    min_ocr_conf=0.05,
    mode="circle_first",
)
cps = result.controls
print("start:", result.start)
print("numbers:", sorted(c.number for c in cps))
for c in cps:
    print(c)

preview = draw_controls(rgb, result)
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(rgb); ax[0].set_title("input"); ax[0].axis("off")
ax[1].imshow(preview); ax[1].set_title(f"controls ({len(cps)}) + start"); ax[1].axis("off")
plt.tight_layout(); plt.show()


## VLM (Yandex Cloud / OpenAI)

По умолчанию **Yandex** (`gemma-3-27b-it`). Нужны `YC_API_KEY` и `YC_FOLDER_ID` в `.env`.
После правки кода — **Kernel → Restart**. При 403: роль `ai.languageModels.user` на каталоге + API-ключ сервисного аккаунта.


In [ ]:
# Ключи Yandex (или положите в .env — не коммитьте ключ в git):
# import os
# os.environ["YC_API_KEY"] = "..."
# os.environ["YC_FOLDER_ID"] = "b1g..."

# Сначала посмотрите, какие модели доступны (gemma-3 часто уже нет):
from src.controls import list_yandex_chat_models
print("\n".join(list_yandex_chat_models()))

# Нужна именно VISION/VL модель. Если в списке только yandexgpt/qwen3/gpt-oss —
# multimodal в каталоге нет, используйте detect_controls(... mode="circle_first").
vlm = detect_controls_vlm(
    rgb,
    provider="yandex",
    model="gemma-3-27b-it",  # замените, если в --list-models появится VL
    refine_circles=True,
    min_confidence=0.35,
)
print("start:", vlm.start)
print("numbers:", sorted(c.number for c in vlm.controls))
for c in vlm.controls:
    print(c)

preview_vlm = draw_controls(rgb, vlm)
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(rgb); ax[0].set_title("input"); ax[0].axis("off")
ax[1].imshow(preview_vlm); ax[1].set_title(f"VLM controls ({len(vlm)})"); ax[1].axis("off")
plt.tight_layout(); plt.show()
